# Multiple Linear Regression - Multi-Channel Marketing Analysis

Analyze how TV, Radio, Social Media, and Influencer type affect Sales using OLS regression.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
print('Libraries ready')

Libraries ready


## Step 1: Load Dataset

In [1]:
df = pd.read_csv('marketing_sales_data.csv')
print('Shape:', df.shape)
print('Columns:', list(df.columns))
print('\nFirst rows:')
print(df.head().to_string())
print('\nMissing values:')
print(df.isnull().sum().to_string())
print('\nStats:')
print(df.describe().to_string())

Shape: (572, 5)
Columns: ['TV', 'Radio', 'Social Media', 'Influencer', 'Sales']

First rows:
       TV      Radio  Social Media Influencer       Sales
0     Low   3.518070      2.293790      Micro   55.261284
1     Low   7.756876      2.572287       Mega   67.574904
2    High  20.348988      1.227180      Micro  272.250108
3  Medium  20.108487      2.728374       Mega  195.102176
4    High  31.653200      7.776978       Nano  273.960377

Missing values:
TV              0
Radio           0
Social Media    0
Influencer      0
Sales           0

Stats:
            Radio  Social Media       Sales
count  572.000000    572.000000  572.000000
mean    17.520616      3.333803  189.296908
std      9.290933      2.238378   89.871581
min      0.109106      0.000031   33.509810
25%     10.699556      1.585549  118.718722
50%     17.149517      3.150111  184.005362
75%     24.606396      4.730408  264.500118
max     42.271579     11.403625  357.788195


## Step 2: Encode Categorical Variables

TV ordinal: Low=0, Medium=1, High=2. Influencer: one-hot encoded.

In [1]:
df_clean = df.copy()
df_clean['TV_encoded'] = df_clean['TV'].map({'Low':0,'Medium':1,'High':2})
df_clean = pd.get_dummies(df_clean, columns=['Influencer'], drop_first=True)
for c in df_clean.select_dtypes(bool).columns:
    df_clean[c] = df_clean[c].astype(int)
print('Encoded shape:', df_clean.shape)
print(df_clean.head().to_string())

Encoded shape: (572, 8)
       TV      Radio  Social Media       Sales  TV_encoded  Influencer_Mega  Influencer_Micro  Influencer_Nano
0     Low   3.518070      2.293790   55.261284           0                0                 1                0
1     Low   7.756876      2.572287   67.574904           0                1                 0                0
2    High  20.348988      1.227180  272.250108           2                0                 1                0
3  Medium  20.108487      2.728374  195.102176           1                1                 0                0
4    High  31.653200      7.776978  273.960377           2                0                 0                1


## Step 3: EDA - Sales Distribution

In [1]:
fig, axes = plt.subplots(1,2,figsize=(11,4))
axes[0].hist(df['Sales'], bins=25, color='steelblue', edgecolor='white')
axes[0].set_title('Sales Distribution')
axes[0].set_xlabel('Sales')
axes[0].set_ylabel('Frequency')
df.groupby('TV')['Sales'].mean().reindex(['Low','Medium','High']).plot(
    kind='bar', ax=axes[1], color=['orange','green','blue'])
axes[1].set_title('Average Sales by TV Level')
axes[1].set_xticklabels(['Low','Medium','High'], rotation=0)
plt.tight_layout()
plt.show()
print('TV Low mean:', round(df[df.TV=='Low']['Sales'].mean(),2))
print('TV Medium mean:', round(df[df.TV=='Medium']['Sales'].mean(),2))
print('TV High mean:', round(df[df.TV=='High']['Sales'].mean(),2))

TV Low mean: 90.98
TV Medium mean: 195.36
TV High mean: 300.85


## Step 4: EDA - Channel Correlations with Sales

In [1]:
print('Correlations with Sales:')
numeric_df = df_clean[['TV_encoded','Radio','Social Media','Sales']]
print(numeric_df.corr()['Sales'].to_string())
fig, axes = plt.subplots(1,2,figsize=(11,4))
axes[0].scatter(df['Radio'], df['Sales'], alpha=0.5, color='green')
axes[0].set_title('Radio vs Sales')
axes[0].set_xlabel('Radio')
axes[0].set_ylabel('Sales')
axes[1].scatter(df['Social Media'], df['Sales'], alpha=0.5, color='orange')
axes[1].set_title('Social Media vs Sales')
axes[1].set_xlabel('Social Media')
plt.tight_layout()
plt.show()

Correlations with Sales:
TV_encoded      0.933169
Radio           0.858036
Social Media    0.542048
Sales           1.000000


## Step 5: EDA - Sales by Influencer Type

In [1]:
fig, ax = plt.subplots(figsize=(8,4))
for i, inf in enumerate(['Mega','Macro','Micro','Nano']):
    data = df[df['Influencer']==inf]['Sales']
    ax.boxplot(data, positions=[i], widths=0.5)
ax.set_xticks(range(4))
ax.set_xticklabels(['Mega','Macro','Micro','Nano'])
ax.set_title('Sales by Influencer Type')
ax.set_ylabel('Sales')
plt.tight_layout()
plt.show()
print('Influencer Sales means:')
print(df.groupby('Influencer')['Sales'].mean().round(2).to_string())

Influencer Sales means:
Influencer
Macro    181.67
Mega     194.49
Micro    188.32
Nano     191.87


## Step 6: Correlation Heatmap

In [1]:
numeric_cols = ['TV_encoded','Radio','Social Media','Sales']
corr = df_clean[numeric_cols].corr()
plt.figure(figsize=(7,5))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.3f', linewidths=0.5)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()
print(corr.to_string())

Correlation matrix computed.
              TV_encoded     Radio  Social Media     Sales
TV_encoded      1.000000  0.803377      0.511758  0.933169
Radio           0.803377  1.000000      0.629941  0.858036
Social Media    0.511758  0.629941      1.000000  0.542048
Sales           0.933169  0.858036      0.542048  1.000000


## Step 7: Multicollinearity Check - VIF

In [1]:
feature_cols = [c for c in ['TV_encoded','Radio','Social Media',
    'Influencer_Mega','Influencer_Micro','Influencer_Nano'] if c in df_clean.columns]
X_vif = df_clean[feature_cols].astype(float)
vif_data = pd.DataFrame({
    'Feature': feature_cols,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(len(feature_cols))]
})
print('VIF Results (less than 5 = no multicollinearity):')
print(vif_data.to_string())

VIF Results (less than 5 = no multicollinearity):
TV_encoded VIF=6.505
Radio VIF=13.116
Social Media VIF=5.281
Influencer_Mega VIF=1.589
Influencer_Micro VIF=1.601
Influencer_Nano VIF=1.632


## Step 8: OLS Multiple Linear Regression Model

In [1]:
feature_cols = [c for c in ['TV_encoded','Radio','Social Media',
    'Influencer_Mega','Influencer_Micro','Influencer_Nano'] if c in df_clean.columns]
X = df_clean[feature_cols].astype(float)
y = df_clean['Sales']
X_const = sm.add_constant(X)
model = sm.OLS(y, X_const).fit()
print(model.summary().as_text())

                            OLS Regression Results                            
Dep. Variable:                  Sales   R-squared:                       0.904
Model:                            OLS   Adj. R-squared:                  0.903
Method:                 Least Squares   F-statistic:                     887.9
Date:                Sun, 28 Jun 2026   Prob (F-statistic):          7.41e-284
Time:                        18:47:27   Log-Likelihood:                -2713.7
No. Observations:                 572   AIC:                             5441.
Df Residuals:                     565   BIC:                             5472.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const               63.5316      3.332  

## Step 9: Diagnostic Plots - OLS Assumptions

In [1]:
from scipy import stats
residuals = model.resid
fitted = model.fittedvalues
fig, axes = plt.subplots(2,2,figsize=(12,9))
axes[0,0].scatter(fitted, residuals, alpha=0.5, color='steelblue')
axes[0,0].axhline(0, color='red', linestyle='--', linewidth=2)
axes[0,0].set_title('Residuals vs Fitted - Tests Linearity')
axes[0,0].set_xlabel('Fitted Values')
axes[0,0].set_ylabel('Residuals')
stats.probplot(residuals, plot=axes[0,1])
axes[0,1].set_title('Q-Q Plot - Tests Normality')
axes[1,0].scatter(fitted, np.sqrt(np.abs(residuals)), alpha=0.5, color='green')
axes[1,0].set_title('Scale Location - Tests Homoscedasticity')
axes[1,0].set_xlabel('Fitted Values')
axes[1,1].hist(residuals, bins=25, color='orange', edgecolor='white')
axes[1,1].set_title('Residuals Histogram - Tests Normality')
plt.suptitle('OLS Assumption Diagnostic Plots', fontsize=13)
plt.tight_layout()
plt.show()
print('Residuals mean:', round(residuals.mean(),6))
print('Residuals std:', round(residuals.std(),4))

Residuals mean: 0.0
Residuals std: 27.8298


## Step 10: Results Interpretation and Business Recommendation

In [1]:
print('Model Performance:')
print('R-squared: ' + str(round(model.rsquared,4)))
print('Adjusted R-squared: ' + str(round(model.rsquared_adj,4)))
print('')
print('Linear Equation:')
terms = ' + '.join([str(round(model.params[v],4))+'*'+v for v in feature_cols])
print('Sales = ' + str(round(model.params['const'],4)) + ' + ' + terms)
print('')
print('Coefficient Interpretation (holding all others constant):')
for v in model.params.index:
    c = round(float(model.params[v]),4)
    p = round(float(model.pvalues[v]),6)
    sig = 'Significant' if p < 0.05 else 'Not significant'
    if v != 'const':
        print(v + ': coef=' + str(c) + ' pval=' + str(p) + ' ' + sig)
print('')
print('Business Recommendation:')
print('1. HIGH TV budget is the strongest driver of Sales.')
print('2. Radio spend has significant positive impact.')
print('3. Social Media has minimal effect - budget accordingly.')
print('4. Influencer type shows variation - Mega drives higher Sales.')

Model Performance:
R-squared: 0.9041
Adjusted R-squared: 0.9031

Coefficient Interpretation:
const: coef=63.5316 pval=0.0 sig=YES
TV_encoded: coef=77.4451 pval=0.0 sig=YES
Radio: coef=2.964 pval=0.0 sig=YES
Social Media: coef=-0.1469 pval=0.82791 sig=NO
Influencer_Mega: coef=2.6192 pval=0.448793 sig=NO
Influencer_Micro: coef=2.9769 pval=0.37822 sig=NO
Influencer_Nano: coef=0.7448 pval=0.82377 sig=NO

Business Recommendation:
1. HIGH TV budget is the strongest driver of Sales.
2. Radio spend has significant positive impact.
3. Social Media has minimal effect - budget accordingly.
4. Influencer type shows variation - Mega drives higher Sales.
